In [6]:
%load_ext autoreload
%autoreload 2
import sys
sys.setrecursionlimit(10000)
import numpy as np
import nerfax

from bio_datasets.structure.parsing import load_structure
from bio_datasets.structure.protein.internal_coordinates import get_backbone_internals, load_backbone_coord_array
from bio_datasets.compress.protein import ProteinBackboneCompressor
from bio_datasets.compress import HistogramEncoding

%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
PDB_FILE = "../tests/AF-Q9R172-F1-model_v4.pdb"  # a long protein chain from AFDB

In [8]:
def load_coords(pdb_file: str, chain: str = "A"):
    """Load cartesian coordinates and internal coordinates for a protein backbone from a PDB file."""
    struct = load_structure(pdb_file)
    struct = struct[struct.chain_id == chain]
    xyz_bb = load_backbone_coord_array(struct)
    print(xyz_bb.shape)
    internals = get_backbone_internals(xyz_bb)
    return xyz_bb, internals

xyz_bb, internals = load_coords(PDB_FILE)
bond_lengths, bond_angles, dihedrals = internals

(2319, 3, 3)


In [9]:

class Compressor:

    def compress(self, internals: tuple[np.ndarray, np.ndarray, np.ndarray]) -> bytes:
        """Compress the internal coordinates. Return bytes."""
        bond_lengths, bond_angles, dihedrals = internals
        # YOUR CODE GOES HERE
        pass
    
    def decompress(self, compressed: bytes) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Decompress the internal coordinates. Return a tuple of the form (bond_lengths, bond_angles, dihedrals)"""
        # YOUR CODE GOES HERE
        pass


def round_trip_rmsd(compressor: Compressor):
    byte_lengths = []
    rmsds = []
    for pdb_file in [
        "../tests/AF-Q9R172-F1-model_v4.pdb",
        "../tests/AF-V9HVX0-F1-model_v4.pdb",
        "../tests/1qys.pdb",
        "../tests/1aq1.pdb",
    ]:
        xyz_bb, internals = load_coords(pdb_file, chain="A")
        compressed_internals = compressor.compress(internals)
        decompressed_internals = compressor.decompress(compressed_internals)
        xyz_bb_recons = nerfax.reconstruct.reconstruct_from_internal_coordinates(
            *decompressed_internals,
            mode='fully_sequential'
        )
        xyz_bb_recons = nerfax.utils.get_align_rigid_bodies_fn(xyz_bb_recons, xyz_bb.reshape((-1,3)))(xyz_bb_recons)
        pdb_id = pdb_file.split("/")[-1].split(".")[0]
        rmsd = nerfax.foldcomp_tests.compute_rmsd(xyz_bb_recons, xyz_bb.reshape((-1,3)))
        print(f"RMSD {pdb_id} ({xyz_bb.shape[0]} residues) {rmsd:.2f} Angstrom; {len(compressed_internals)} bytes")
        rmsds.append(rmsd)
        byte_lengths.append(len(compressed_internals))
    return byte_lengths, rmsds

Foldcomp uses 8 bytes per backbone residue and 1 byte for all sidechain atoms.
It'd be cool to be able to reproduce their sidechain encoding / decoding.
(And like Jude said to infer the amino acid identity from that directly)

Foldcomp assumes idealised bond lengths, and has to compensate with error-correction
during decoding (the step performed by nerfax.reconstruct.reconstruct_from_internal_coordinates)
It also stores full backbone coordinates for every 25th residue, meaning that effectively ~9.5 bytes per residue

It's possible with a naive discretisation of each of bond lengths, bond angles, and torsion
angles to get a decent 12 byte encoding.

Does foldcomp encode all 3 torsion angles or just phi and psi and not omega?

Target RMSD <0.1 A (averaged across some typical proteins)

In [10]:
class WrappedBackboneCompressor(Compressor):
    def __init__(self, compressor: ProteinBackboneCompressor):
        self.compressor = compressor

    def compress(self, internals: tuple[np.ndarray, np.ndarray, np.ndarray]) -> bytes:
        bond_lengths, bond_angles, dihedrals = internals
        return self.compressor.compress_internals(bond_lengths, bond_angles, dihedrals)

    def decompress(self, compressed: bytes) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        return self.compressor.decompress_internals(compressed)


bond_length_encoders = [
    HistogramEncoding.build(bond_lengths[:, i], 512) for i in range(3)
]

bond_angle_encoders = [
    HistogramEncoding.build(bond_angles[:, i], 8192) for i in range(3)
]

dihedral_encoders = [
    HistogramEncoding.build(dihedrals[:, i], 16384, low=-np.pi, high=np.pi) for i in range(3)
]



In [11]:
round_trip_rmsd(
    WrappedBackboneCompressor(
        ProteinBackboneCompressor(
            bond_length_encoders,
            bond_angle_encoders,
            dihedral_encoders
        )
    )
)

(2319, 3, 3)
RMSD AF-Q9R172-F1-model_v4 (2319 residues) 1.33 Angstrom; 23528 bytes
(61, 3, 3)
RMSD AF-V9HVX0-F1-model_v4 (61 residues) 0.06 Angstrom; 28037 bytes
(92, 3, 3)
RMSD 1qys (92 residues) 0.12 Angstrom; 67307 bytes
(277, 3, 3)
RMSD 1aq1 (277 residues) 25.86 Angstrom; 184977 bytes


([23528, 28037, 67307, 184977],
 [Array(1.3276308, dtype=float32),
  Array(0.05543357, dtype=float32),
  Array(0.12070048, dtype=float32),
  Array(25.860868, dtype=float32)])

TODO: test whether it helps to add sparsity to bond lengths,
offset to angles.